# 🚗 Getaround — Analyse des Retards & Politique de Délai Minimum

**Contexte :** Getaround souhaite instaurer un délai minimum entre deux locations consécutives pour réduire les conflits liés aux retours tardifs.  
**Objectif :** Aider le chef de produit à choisir le bon **seuil** (durée du délai) et la bonne **portée** (toutes les voitures ou uniquement les voitures connectées).

---

## 1. 📦 Imports & Chargement des Données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

df = pd.read_excel('get_around_delay_analysis.xlsx')
print(f"Dataset : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
df.head()

## 2. 🔍 Exploration Générale

In [ ]:
print("Types et valeurs manquantes :")
missing = pd.DataFrame({
    'dtype': df.dtypes,
    'missing': df.isnull().sum(),
    'missing_%': (df.isnull().sum() / len(df) * 100).round(1)
})
print(missing)

print("\ncheckin_type :")
print(df['checkin_type'].value_counts())
print("\nstate :")
print(df['state'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribution checkin_type
ct = df['checkin_type'].value_counts()
axes[0].pie(ct, labels=ct.index, autopct='%1.1f%%', colors=['#3498db','#2ecc71'],
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0].set_title('Type de check-in', fontweight='bold')

# Distribution state
st = df['state'].value_counts()
axes[1].pie(st, labels=st.index, autopct='%1.1f%%', colors=['#27ae60','#e74c3c'],
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('État de la location', fontweight='bold')

# Annulations par type
cancel_rate = df.groupby('checkin_type').apply(
    lambda x: (x['state'] == 'canceled').sum() / len(x) * 100)
bars = axes[2].bar(cancel_rate.index, cancel_rate.values,
                   color=['#3498db','#2ecc71'], edgecolor='white', linewidth=1.5)
axes[2].set_title('Taux d\'annulation par type', fontweight='bold')
axes[2].set_ylabel('Taux d\'annulation (%)')
axes[2].yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(bars, cancel_rate.values):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 3. ⏱️ Analyse des Retards au Checkout

In [ ]:
# On travaille sur les locations terminées avec délai connu
df_ended = df[df['state'] == 'ended'].copy()
df_delay = df_ended.dropna(subset=['delay_at_checkout_in_minutes']).copy()

total_with_delay = len(df_delay)
n_late = (df_delay['delay_at_checkout_in_minutes'] > 0).sum()
n_early = (df_delay['delay_at_checkout_in_minutes'] <= 0).sum()

print(f"Locations avec info retard : {total_with_delay:,}")
print(f"  → En retard  : {n_late:,} ({n_late/total_with_delay:.1%})")
print(f"  → À l'heure ou en avance : {n_early:,} ({n_early/total_with_delay:.1%})")

print("\nStatistiques du retard (minutes) :")
print(df_delay['delay_at_checkout_in_minutes'].describe().round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution globale (zoom entre -200 et 500 min pour lisibilité)
zoom = df_delay[(df_delay['delay_at_checkout_in_minutes'] >= -200) &
                (df_delay['delay_at_checkout_in_minutes'] <= 500)]
axes[0].hist(zoom['delay_at_checkout_in_minutes'], bins=60,
             color='#3498db', edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='#e74c3c', linestyle='--', lw=2, label='Heure limite')
axes[0].axvline(df_delay['delay_at_checkout_in_minutes'].median(),
                color='#f39c12', linestyle='--', lw=2,
                label=f"Médiane : {df_delay['delay_at_checkout_in_minutes'].median():.0f} min")
axes[0].set_title('Distribution des retards au checkout\n(zoom entre -200 et +500 min)', fontweight='bold')
axes[0].set_xlabel('Retard (minutes)') ; axes[0].set_ylabel('Nombre de locations')
axes[0].legend()

# Par type de check-in
for ctype, color in [('mobile', '#3498db'), ('connect', '#2ecc71')]:
    subset = df_delay[df_delay['checkin_type'] == ctype]['delay_at_checkout_in_minutes']
    subset_zoom = subset[(subset >= -200) & (subset <= 500)]
    axes[1].hist(subset_zoom, bins=50, alpha=0.6, label=ctype, color=color, edgecolor='white')
axes[1].axvline(0, color='#e74c3c', linestyle='--', lw=2)
axes[1].set_title('Retards par type de check-in\n(zoom -200 à +500 min)', fontweight='bold')
axes[1].set_xlabel('Retard (minutes)') ; axes[1].set_ylabel('Nombre de locations')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Médiane et % de retardataires par type
stats_by_type = df_delay.groupby('checkin_type').agg(
    n_locations=('delay_at_checkout_in_minutes', 'count'),
    mediane_retard=('delay_at_checkout_in_minutes', 'median'),
    moyenne_retard=('delay_at_checkout_in_minutes', 'mean'),
    pct_en_retard=('delay_at_checkout_in_minutes', lambda x: (x > 0).mean() * 100)
).round(1)
print("Statistiques de retard par type de check-in :")
print(stats_by_type)

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
groups = df_delay.groupby('checkin_type')['delay_at_checkout_in_minutes']
data = [groups.get_group(g).dropna().clip(-200, 500) for g in ['mobile', 'connect']]
bp = ax.boxplot(data, labels=['Mobile', 'Connect'], patch_artist=True,
                medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], ['#3498db', '#2ecc71']):
    patch.set_facecolor(color) ; patch.set_alpha(0.7)
ax.axhline(0, color='#e74c3c', linestyle='--', lw=1.5, label='Heure limite')
ax.set_title('Boxplot des retards par type\n(clippé à ±200min)', fontweight='bold')
ax.set_ylabel('Retard (minutes)') ; ax.legend()

ax2 = axes[1]
pct = stats_by_type['pct_en_retard']
bars = ax2.bar(pct.index, pct.values, color=['#3498db', '#2ecc71'], edgecolor='white', lw=1.5)
ax2.set_title('% de conducteurs en retard\npar type de check-in', fontweight='bold')
ax2.set_ylabel('%') ; ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(bars, pct.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}%', ha='center', fontweight='bold')

plt.tight_layout() ; plt.show()

**Observation clé :** Les voitures **mobile** ont un taux de retard significativement plus élevé (~64%) et une médiane de retard positive (+14 min). Les voitures **connect** sont plus souvent rendues en avance (médiane -9 min), probablement parce que le processus sans rencontre réduit les frictions.

## 4. 🔗 Impact sur la Location Suivante

In [ ]:
# Locations avec une location précédente identifiée
df_consec = df.dropna(subset=['previous_ended_rental_id',
                               'time_delta_with_previous_rental_in_minutes']).copy()

# Joindre les infos du retard de la location précédente
prev_delay = df[['rental_id', 'delay_at_checkout_in_minutes']].rename(
    columns={'rental_id': 'previous_ended_rental_id',
             'delay_at_checkout_in_minutes': 'prev_delay_minutes'})
df_consec = df_consec.merge(prev_delay, on='previous_ended_rental_id', how='left')

# Cas impactés : le retard précédent > le delta entre les deux locations
df_consec['is_impacted'] = (
    df_consec['prev_delay_minutes'] > df_consec['time_delta_with_previous_rental_in_minutes']
).fillna(False)

n_consec   = len(df_consec)
n_impacted = df_consec['is_impacted'].sum()
print(f"Locations avec une précédente : {n_consec}")
print(f"Impactées par un retard précédent : {n_impacted} ({n_impacted/n_consec:.1%})")

# Répartition par type
print("\nImpact par type de check-in :")
print(df_consec.groupby('checkin_type')['is_impacted'].agg(['sum','mean']).rename(
    columns={'sum': 'n_impactées', 'mean': 'taux'}).assign(
    taux=lambda x: x['taux'].map('{:.1%}'.format)))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Distribution du delta entre deux locations
axes[0].hist(df_consec['time_delta_with_previous_rental_in_minutes'].clip(0, 720),
             bins=40, color='#9b59b6', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribution du délai entre locations consécutives', fontweight='bold')
axes[0].set_xlabel('Délai (minutes)') ; axes[0].set_ylabel('Nombre de cas')
for t, ls in [(60,'--'),(120,'-.'),(180,':')]:
    axes[0].axvline(t, color='#e74c3c', linestyle=ls, lw=1.5, label=f'{t} min')
axes[0].legend(title='Seuils potentiels')

# Retard précédent vs delta — scatter
impacted = df_consec[df_consec['is_impacted']]
not_imp  = df_consec[~df_consec['is_impacted']]
clip_val = 500
axes[1].scatter(not_imp['time_delta_with_previous_rental_in_minutes'].clip(0, clip_val),
                not_imp['prev_delay_minutes'].clip(0, clip_val),
                alpha=0.3, s=15, color='#2ecc71', label='Non impactées')
axes[1].scatter(impacted['time_delta_with_previous_rental_in_minutes'].clip(0, clip_val),
                impacted['prev_delay_minutes'].clip(0, clip_val),
                alpha=0.6, s=20, color='#e74c3c', label='Impactées')
axes[1].plot([0, clip_val], [0, clip_val], 'k--', lw=1.5, label='Retard = Delta')
axes[1].set_xlabel('Délai entre locations (min)') ; axes[1].set_ylabel('Retard location précédente (min)')
axes[1].set_title('Retard précédent vs Délai entre locations\n(clippé à 500 min)', fontweight='bold')
axes[1].legend()

plt.tight_layout() ; plt.show()

## 5. 📐 Simulation des Seuils & Portées

In [ ]:
thresholds = [30, 60, 90, 120, 150, 180, 210, 240, 300, 360, 480, 720]
scopes     = ['all', 'connect']

results = []
total_rentals = len(df)

for scope in scopes:
    df_scope = df if scope == 'all' else df[df['checkin_type'] == 'connect']
    consec_scope = df_consec if scope == 'all' else df_consec[df_consec['checkin_type'] == 'connect']

    for t in thresholds:
        # Locations bloquées (auraient été visibles mais ne l'auraient plus été)
        blocked = (df_scope['time_delta_with_previous_rental_in_minutes'] < t).sum()

        # Cas problématiques résolus
        solved = consec_scope[
            (consec_scope['is_impacted']) &
            (consec_scope['time_delta_with_previous_rental_in_minutes'] < t)
        ]

        total_problematic = consec_scope['is_impacted'].sum()

        results.append({
            'Portée': scope,
            'Seuil (min)': t,
            'Locations bloquées': blocked,
            '% revenus affectés': blocked / total_rentals * 100,
            'Cas résolus': len(solved),
            '% cas résolus': len(solved) / total_problematic * 100 if total_problematic > 0 else 0
        })

results_df = pd.DataFrame(results)
print(results_df.round(1).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
colors = {'all': '#e74c3c', 'connect': '#3498db'}
labels = {'all': 'Toutes les voitures', 'connect': 'Connect uniquement'}

for scope in scopes:
    sub = results_df[results_df['Portée'] == scope]
    c = colors[scope] ; l = labels[scope]

    axes[0,0].plot(sub['Seuil (min)'], sub['Locations bloquées'],
                   marker='o', color=c, label=l, lw=2)
    axes[0,1].plot(sub['Seuil (min)'], sub['% revenus affectés'],
                   marker='o', color=c, label=l, lw=2)
    axes[1,0].plot(sub['Seuil (min)'], sub['Cas résolus'],
                   marker='o', color=c, label=l, lw=2)
    axes[1,1].plot(sub['Seuil (min)'], sub['% cas résolus'],
                   marker='o', color=c, label=l, lw=2)

titles = ['Locations bloquées', '% Revenus potentiellement affectés',
          'Cas problématiques résolus', '% Cas résolus']
ylabels = ['Nombre', '%', 'Nombre', '%']

for ax, title, ylabel in zip(axes.flat, titles, ylabels):
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Seuil (minutes)') ; ax.set_ylabel(ylabel)
    ax.legend() ; ax.grid(alpha=0.3)
    if '%' in ylabel:
        ax.yaxis.set_major_formatter(mtick.PercentFormatter())

plt.suptitle('Simulation : Impact du Seuil et de la Portée', fontsize=14,
             fontweight='bold', y=1.02)
plt.tight_layout() ; plt.show()

In [ ]:
# Ratio efficacité : cas résolus / locations bloquées
results_df['Efficacité (résolus/bloqués)'] = (
    results_df['Cas résolus'] / results_df['Locations bloquées'].replace(0, np.nan)
).round(3)

print("📊 Tableau décisionnel — Efficacité par seuil et portée :")
pivot = results_df.pivot_table(
    index='Seuil (min)', columns='Portée',
    values=['Locations bloquées', 'Cas résolus', 'Efficacité (résolus/bloqués)']
)
print(pivot.round(2))

## 6. 💡 Recommandations

In [ ]:
# Visualisation synthétique : courbe coût-bénéfice
fig, ax = plt.subplots(figsize=(10, 6))

for scope in scopes:
    sub = results_df[results_df['Portée'] == scope]
    sc = ax.scatter(sub['% revenus affectés'], sub['% cas résolus'],
                    c=sub['Seuil (min)'], cmap='YlOrRd', s=120,
                    edgecolors=colors[scope], linewidth=2, zorder=5,
                    label=labels[scope])
    for _, row in sub.iterrows():
        ax.annotate(f"{int(row['Seuil (min)'])}min",
                    (row['% revenus affectés'], row['% cas résolus']),
                    textcoords='offset points', xytext=(5, 4), fontsize=7)

plt.colorbar(sc, ax=ax, label='Seuil (minutes)')
ax.set_xlabel('% Locations potentiellement bloquées (coût)', fontweight='bold')
ax.set_ylabel('% Cas problématiques résolus (bénéfice)', fontweight='bold')
ax.set_title('Compromis Coût / Bénéfice par Seuil et Portée', fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend() ; ax.grid(alpha=0.3)

# Zone recommandée
ax.axhspan(55, 95, alpha=0.05, color='green', label='Zone recommandée')
plt.tight_layout() ; plt.show()

## 7. 📝 Conclusion & Recommandation Finale

### Résultats clés

| Indicateur | Valeur |
|---|---|
| Locations totales | 21 310 |
| En retard au checkout | ~57% des locations terminées |
| Médiane du retard (mobile) | +14 min |
| Médiane du retard (connect) | -9 min |
| Locations avec une suivante | 1 841 |
| Impactées par un retard précédent | 270 (14.7%) |

### Recommandation

**Seuil recommandé : 60 minutes — Portée : Connect uniquement**

- Un seuil de **60 minutes** avec une portée **Connect** bloque seulement ~181 locations supplémentaires (~1% du total) tout en résolvant **~65% des cas problématiques** identifiés sur ce segment.
- Les voitures **Mobile** ont une médiane de retard élevée, mais imposer un seuil global bloquerait trop de revenus. Une **phase pilote sur Connect** permet de valider l'impact avant un éventuel déploiement global.
- Si les résultats sont positifs sur Connect, passer à un seuil de **120 minutes** pour **all** résoudrait ~83% des cas pour un coût de ~3% du total des locations.

### Prochaines étapes
1. Tester le seuil 60min sur Connect (phase pilote)
2. Mesurer l'impact réel sur les annulations et la satisfaction client
3. Étendre progressivement à Mobile selon les résultats